# 0. Submission information

**Submission date:** 16 November 2026

**QPM 2026-2027: Assignment 2**

**Group members:** LOLLI Alessandro, PERALDI Leandrè, MASTROGIACOMO Paolo

**Comments/challenges:** We reuse the monthly returns saved in Assignment 1 (the notebook re-downloads them if the file is not found). Constrained portfolios are computed numerically with `scipy.optimize.minimize` (SLSQP). We used Claude (Anthropic) to help structure and debug the code; all results and interpretations were checked by the group.

### Setup and data preparation
*The data are the same as in the previous assignment: S&P 500 constituents from Wikipedia, start date "2000-01-01", end date "2022-12-31", adjusted prices, dropping all-NaN columns and companies with more than 100 missing observations.*

The cell below loads the monthly returns saved in Assignment 1 (`sp500_monthly_returns.csv`). If the file is not in the current folder (or in `../Assignment-1/`), it rebuilds the dataset with exactly the steps of Assignment 1.

In [ ]:
# !pip install -q yfinance

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

pd.set_option("display.precision", 4)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (9, 4)

def build_sp500_returns(start_date="2000-01-01", end_date="2022-12-31"):
    # Same steps as Assignment 1, Q2.1-Q2.2
    import requests
    from io import StringIO
    import yfinance as yf
    try:
        html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                            headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
        table = pd.read_html(StringIO(html))[0]
    except Exception:
        table = pd.read_csv("https://raw.githubusercontent.com/datasets/"
                            "s-and-p-500-companies/main/data/constituents.csv")
    tickers = table["Symbol"].astype(str).str.replace(".", "-", regex=False).tolist()
    daily = yf.download(tickers=tickers, start=start_date, end=end_date,
                        auto_adjust=False, progress=False)["Adj Close"]
    monthly = daily.resample("ME").last()
    monthly = monthly.dropna(axis=1, how="all")
    monthly = monthly.loc[:, monthly.isna().sum() <= 100]
    return monthly.pct_change(fill_method=None).iloc[1:]

candidates = ["sp500_monthly_returns.csv", "../Assignment-1/sp500_monthly_returns.csv"]
path = next((p for p in candidates if os.path.exists(p)), None)
if path is not None:
    ret_all = pd.read_csv(path, index_col=0, parse_dates=True)
    print("Loaded", path)
else:
    print("File not found: rebuilding the dataset (takes a few minutes)")
    ret_all = build_sp500_returns()
    ret_all.to_csv("sp500_monthly_returns.csv")
print("All S&P 500 monthly returns:", ret_all.shape)

# 1. Question 1 (no constraints on the weights)

## Q1.1
*From the data that we used for the previous assignment, select the following 10 companies (these are the first 10 companies with no missing data): "MMM", "AOS", "ABT", "ADM", "ADBE", "ADP", "AES", "AFL", "A", "AKAM". So, our "new" dataset for this assignment will consist of the simple monthly returns you saved in the last assignment, from February 2000 to December 2022, but just for these 10 companies. To reduce the work required for this assignment, please assume that the risk-free rate of return is zero.*

In [ ]:
tickers = ["MMM", "AOS", "ABT", "ADM", "ADBE", "ADP", "AES", "AFL", "A", "AKAM"]
R = ret_all.loc["2000-02":"2022-12", tickers]
assert R.isna().sum().sum() == 0, "missing values in the 10 selected stocks"
T, N = R.shape
print(f"T = {T} months ({R.index[0].date()} to {R.index[-1].date()}), N = {N} stocks, no missing values")

rf = 0.0                      # risk-free rate assumed to be zero
summary_assets = pd.DataFrame({"mean (ann.)": 12 * R.mean(),
                               "vol (ann.)": np.sqrt(12) * R.std(),
                               "Sharpe (ann.)": np.sqrt(12) * R.mean() / R.std()})
summary_assets.round(3)

**Answer.** The dataset has T = 275 monthly returns (February 2000 – December 2022) for the 10 stocks, with no missing values. With R_f = 0, excess returns equal returns, the Sharpe ratio is √12 × mean / standard deviation, and the mean-variance problem with a risk-free asset uses μ − R_f 1 = μ.

## Q1.2
*Choose the estimation window to be T_est = 60 months of monthly returns. Call this the estimation sample. Use the estimation sample to compute the following two portfolio strategies: (a) mean-variance portfolio (MVP) without constraints on the size of the weight (assume that a risk-free asset is available, with the risk-free rate equal to zero); (b) global minimum variance (GMV) portfolio without constraints on the size of the weight. For each of the two portfolios, rescale the weights in the risky assets so that they sum to 1; that is, you are "fully invested" in just the risky assets.*

In [ ]:
T_est = 60
ones = np.ones(N)

def w_mvp(mu, V):
    # Mean-variance weights (1/gamma) V^{-1}(mu - rf 1), rescaled to sum to 1.
    # gamma cancels after rescaling, so this is the tangency portfolio.
    x = np.linalg.solve(V, mu - rf * ones)
    return x / x.sum()

def w_gmv(V):
    # Global minimum variance: V^{-1} 1 / (1' V^{-1} 1)
    x = np.linalg.solve(V, ones)
    return x / x.sum()

est = R.iloc[:T_est]                         # estimation sample: first 60 months
mu_hat = est.mean().values                   # sample means
V_hat = est.cov().values                     # sample covariance matrix
print("Estimation sample:", est.index[0].date(), "to", est.index[-1].date())
print("Condition number of the sample covariance matrix:", round(np.linalg.cond(V_hat), 1))

w0 = pd.DataFrame({"MVP": w_mvp(mu_hat, V_hat), "GMV": w_gmv(V_hat)}, index=tickers)
display(w0.round(3))
print("Sum of weights:", w0.sum().round(6).to_dict())
print("Sum of the raw MVP weights 1'V^{-1}mu =", round(np.linalg.solve(V_hat, mu_hat).sum(), 4))

**Answer.** Using the first 60 months (February 2000 – January 2005) we estimate μ̂ (sample means) and V̂ (sample covariance matrix, divided by T − 1) and compute:

* **MVP**: w = (1/γ) V̂⁻¹ μ̂ (R_f = 0), rescaled so that the risky weights sum to one, w_MVP = V̂⁻¹μ̂ / 1'V̂⁻¹μ̂. The risk aversion γ cancels in the rescaling, so this is the **tangency portfolio** (Lecture 3).
* **GMV**: w_GMV = V̂⁻¹1 / 1'V̂⁻¹1, which does not use expected returns at all (Lecture 4).

Both portfolios contain short positions (negative weights), because no constraints are imposed. The MVP weights are typically more extreme than the GMV weights because they depend on the noisy estimates of the means.

## Q1.3
*Now use a rolling window of T_est = 60 months to estimate the portfolio weights for the two strategies listed above for each of the T − T_est months. That is, repeat the calculations of the previous question for all the dates after the first 60 months.*

In [ ]:
def rolling_weights(R, rule, T_est=60):
    # At each month t = T_est, ..., T-1 the weights use only months t-T_est, ..., t-1
    # (Python slices exclude the end, so month t is NOT in the estimation window: no look-ahead).
    W = []
    for t in range(T_est, len(R)):
        window = R.iloc[t - T_est:t]
        W.append(rule(window.mean().values, window.cov().values))
    return pd.DataFrame(W, index=R.index[T_est:], columns=R.columns)

W_mvp = rolling_weights(R, lambda mu, V: w_mvp(mu, V))
W_gmv = rolling_weights(R, lambda mu, V: w_gmv(V))
print("Number of rebalancing dates (T - T_est):", len(W_mvp))
print("Row t of the weight tables is used for the return of month t (estimated with months t-60..t-1)")

denom = pd.Series([np.linalg.solve(R.iloc[t-T_est:t].cov().values,
                                   R.iloc[t-T_est:t].mean().values).sum()
                   for t in range(T_est, len(R))], index=W_mvp.index)
print(f"Months in which 1'V^-1 mu < 0 (rescaled MVP flips sign): {(denom < 0).sum()}")
print(f"Smallest |1'V^-1 mu|: {denom.abs().min():.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
W_mvp.plot(ax=ax[0], legend=False, title="MVP weights over time")
W_gmv.plot(ax=ax[1], legend=False, title="GMV weights over time")
ax[1].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

print("Average and extreme weights:")
pd.DataFrame({"avg max |w|": [W_mvp.abs().max(axis=1).mean(), W_gmv.abs().max(axis=1).mean()],
              "largest |w| ever": [W_mvp.abs().values.max(), W_gmv.abs().values.max()]},
             index=["MVP", "GMV"]).round(2)

**Answer.** With a rolling window of 60 months we re-estimate μ̂ and V̂ every month, dropping the oldest observation and adding the newest, and recompute both portfolios: 275 − 60 = 215 sets of weights (first window: Feb 2000 – Jan 2005; first out-of-sample month: Feb 2005).

The plots show that the **MVP weights are much more extreme and unstable** than the GMV weights. One reason is specific to the rescaling: the MVP weights are divided by 1'V̂⁻¹μ̂. When this number is close to zero the rescaled weights explode, and when it is negative the "fully invested" portfolio flips sign (it becomes short the efficient portfolio). The output above reports how often this happens in our data. Both problems come from the estimation error in the means (Lecture 4).

## Q1.4
*Use the time series of portfolio weights for each of the two portfolio strategies to compute the out-of-sample portfolio returns. That is, for each of the two portfolio strategies that you estimate at each date t, compute its out-of-sample return in month t + 1.*

In [ ]:
R_oos = R.iloc[T_est:]                                   # returns of months T_est, ..., T-1
oos1 = pd.DataFrame({"MVP": (W_mvp * R_oos).sum(axis=1),  # w estimated up to t-1 times R_t
                     "GMV": (W_gmv * R_oos).sum(axis=1)})
print("Out-of-sample months:", len(oos1), "from", oos1.index[0].date(), "to", oos1.index[-1].date())
display(oos1.describe().round(4))

(1 + oos1["GMV"]).cumprod().plot(title="Growth of $1 (out of sample)", label="GMV", legend=True)
plt.show()
mvp_wealth = (1 + oos1["MVP"]).cumprod()
print("Final value of $1 invested in MVP:", round(mvp_wealth.iloc[-1], 4),
      "| minimum value reached:", round(mvp_wealth.min(), 4))

**Answer.** For every month t the weights are estimated with the 60 previous months and applied to the returns of month t, i.e., weights chosen at the end of month t − 1 earn the return of the following month. This gives 215 **out-of-sample** returns per strategy (February 2005 – December 2022). No information from the evaluation month is used to choose the weights, so there is no look-ahead bias (Lecture 2).

## Q1.5
*Now, compute the Sharpe ratio of the out-of-sample returns for the two portfolio strategies. Which strategy has the higher Sharpe ratio? Why?*

In [ ]:
def sharpe(r):
    return np.sqrt(12) * r.mean() / r.std(ddof=1)        # rf = 0, annualized

def turnover(W):
    return W.diff().abs().sum(axis=1).iloc[1:].mean()    # average monthly |w_t - w_{t-1}|

def summarize(oos, Ws):
    return pd.DataFrame({
        "mean (ann.)": 12 * oos.mean(),
        "vol (ann.)": np.sqrt(12) * oos.std(ddof=1),
        "Sharpe (ann.)": oos.apply(sharpe),
        "avg max |w|": {k: W.abs().max(axis=1).mean() for k, W in Ws.items()},
        "avg turnover": {k: turnover(W) for k, W in Ws.items()}})

def jkm_test(a, b):
    # Jobson-Korkie test with Memmel correction (Lecture 2), monthly Sharpe ratios
    a, b = np.asarray(a), np.asarray(b)
    Tn = len(a)
    sa, sb = a.mean() / a.std(ddof=1), b.mean() / b.std(ddof=1)
    rho = np.corrcoef(a, b)[0, 1]
    var = (2 - 2 * rho + 0.5 * (sa**2 + sb**2 - 2 * sa * sb * rho**2)) / Tn
    z = (sa - sb) / np.sqrt(var)
    return z, 2 * stats.norm.sf(abs(z))

table1 = summarize(oos1, {"MVP": W_mvp, "GMV": W_gmv})
display(table1.round(3))
best1 = table1["Sharpe (ann.)"].idxmax()
z, p = jkm_test(oos1["GMV"], oos1["MVP"])
print(f"Higher out-of-sample Sharpe ratio: {best1}")
print(f"JKM test GMV vs MVP: z = {z:.2f}, p-value = {p:.3f}")

**Answer.** The table reports the annualized out-of-sample Sharpe ratios (R_f = 0) and the strategy with the higher one; the expected result, consistent with DeMiguel, Garlappi and Uppal (2009) and with Lecture 4, is that **GMV has a higher Sharpe ratio than MVP** (and much lower volatility and turnover). Why:

1. **Estimation error in the means.** The MVP needs the expected returns μ, which are estimated very imprecisely: with a 5-year window and an annual volatility of about 30%, the standard error of each annual mean is about 30%/√5 ≈ 13%, larger than the means themselves. Sampling more often does not help (Lecture 1).
2. **The inverse covariance matrix amplifies these errors.** In w = V̂⁻¹μ̂, small differences in μ̂ between correlated stocks are turned into large long–short positions (Lecture 4). The MVP weights are extreme and unstable, so its out-of-sample returns are very volatile.
3. **Rescaling makes it worse.** Dividing by 1'V̂⁻¹μ̂ produces exploding weights when this number is near zero, and a sign flip (short the tangency portfolio) when it is negative.
4. **GMV ignores the means.** It is equivalent to shrinking all expected returns to the same value (100% shrinkage) and uses only V̂, which is estimated much more precisely than μ̂. It is the minimum-variance portfolio, so its out-of-sample volatility is low and its weights are stable.

The JKM test (Lecture 2) tells us whether the difference in Sharpe ratios is statistically significant; with 215 monthly observations the standard error of each annual Sharpe ratio is about 0.24, so modest differences may not be significant, and the test assumes IID Normal returns.

# 2. Question 2 (non-negativity constraints)

## Q2.1
*Using the estimation window of T_est = 60 months of monthly returns, now compute the following two portfolio strategies: (a) mean-variance portfolio with nonnegativity constraints on the weights (when a risk-free asset is available, and set this rate to 0); we will refer to this portfolio as "MVP-C"; (b) global minimum variance (GMV) portfolio with nonnegativity constraints; we will refer to this portfolio as "GMV-C". For each of the two portfolios, rescale the weights in the risky assets so that they sum to 1; that is, you are "fully invested" in just the risky assets.*

In [ ]:
def w_gmv_c(V):
    # min w'Vw  s.t.  sum(w) = 1, w >= 0   (no closed form: numerical solution)
    cons = {"type": "eq", "fun": lambda w: w.sum() - 1}
    res = minimize(lambda w: w @ V @ w, ones / N, method="SLSQP",
                   bounds=[(0, 1)] * N, constraints=[cons],
                   options={"ftol": 1e-12, "maxiter": 1000})
    w = np.clip(res.x, 0, None)
    return w / w.sum()

def w_mvp_c(mu, V, gamma=1.0):
    # max w'(mu - rf 1) - gamma/2 w'Vw  s.t.  w >= 0 ; the rest is in the risk-free asset.
    # The feasible set {w >= 0} is a cone, so the solution scales with 1/gamma and the
    # rescaled weights do not depend on gamma (long-only maximum-Sharpe portfolio).
    obj = lambda w: gamma / 2 * w @ V @ w - w @ (mu - rf * ones)
    res = minimize(obj, ones / N, method="SLSQP", bounds=[(0, None)] * N,
                   options={"ftol": 1e-12, "maxiter": 1000})
    w = np.clip(res.x, 0, None)
    if w.sum() < 1e-10:
        # All estimated excess means <= 0: the optimum is 100% in the risk-free asset and
        # cannot be rescaled; to stay fully invested we use GMV-C in that month.
        return w_gmv_c(V)
    return w / w.sum()

w0_c = pd.DataFrame({"MVP-C": w_mvp_c(mu_hat, V_hat), "GMV-C": w_gmv_c(V_hat)}, index=tickers)
display(w0_c.round(3))
print("Sum of weights:", w0_c.sum().round(6).to_dict())
print("Number of zero weights:", (w0_c < 1e-6).sum().to_dict())

# Check that gamma does not matter for MVP-C after rescaling
print("MVP-C with gamma = 1 and gamma = 10 identical:",
      np.allclose(w_mvp_c(mu_hat, V_hat, 1), w_mvp_c(mu_hat, V_hat, 10), atol=1e-4))

**Answer.** With short-sale (non-negativity) constraints there is no closed-form solution, so we solve the problems numerically with `scipy.optimize.minimize` (method SLSQP, Lecture 4 appendix):

* **MVP-C**: maximize the mean-variance utility w'μ̂ − (γ/2) w'V̂w subject to w ≥ 0 (the remainder is invested in the risk-free asset at 0%), then rescale the risky weights to sum to one. Because the constraint set {w ≥ 0} is a cone, the solution for any γ is the solution for γ = 1 divided by γ, so after rescaling the weights do not depend on γ (verified above): MVP-C is the long-only maximum-Sharpe portfolio. If all estimated means were non-positive, the optimum would be 100% risk-free and could not be rescaled; in that case we use GMV-C (this is stated in the code).
* **GMV-C**: minimize w'V̂w subject to 1'w = 1 and w ≥ 0 (Jagannathan and Ma 2003).

Both portfolios have several weights exactly equal to zero: the constraint binds for the stocks that the unconstrained portfolios wanted to short.

## Q2.2
*Now use a rolling window of T_est = 60 months to estimate the portfolio weights for the two strategies listed above for each of the T − T_est months. That is, repeat the calculations of the previous question for all the dates after the first 60 months.*

In [ ]:
W_mvpc = rolling_weights(R, lambda mu, V: w_mvp_c(mu, V))
W_gmvc = rolling_weights(R, lambda mu, V: w_gmv_c(V))
print("Number of rebalancing dates:", len(W_mvpc))
print("Average number of zero weights per month:",
      {"MVP-C": round((W_mvpc < 1e-6).sum(axis=1).mean(), 2),
       "GMV-C": round((W_gmvc < 1e-6).sum(axis=1).mean(), 2)})

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
W_mvpc.plot.area(ax=ax[0], legend=False, title="MVP-C weights (stacked)", linewidth=0)
W_gmvc.plot.area(ax=ax[1], legend=False, title="GMV-C weights (stacked)", linewidth=0)
ax[1].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout(); plt.show()

**Answer.** We repeat the constrained optimizations every month with the same rolling 60-month window (215 rebalancing dates). All weights are now between 0 and 1 and sum to one. MVP-C tends to concentrate in the few stocks with the highest recent average returns (many zero weights, frequent changes), while GMV-C spreads the weights more evenly across the low-risk stocks and changes more slowly.

## Q2.3
*Use the time series of portfolio weights for each of the two portfolio strategies to compute the out-of-sample portfolio returns. That is, for each of the two portfolio strategies that you estimate at each date t, compute its out-of-sample return in month t + 1.*

In [ ]:
oos2 = pd.DataFrame({"MVP-C": (W_mvpc * R_oos).sum(axis=1),
                     "GMV-C": (W_gmvc * R_oos).sum(axis=1)})
display(oos2.describe().round(4))
(1 + oos2).cumprod().plot(title="Growth of $1 (out of sample), constrained portfolios")
plt.show()

**Answer.** As in Q1.4, the weights estimated with months t − 60, …, t − 1 are applied to the returns of month t, giving 215 out-of-sample monthly returns for each constrained strategy.

## Q2.4
*Now, compute the Sharpe ratio of the out-of-sample returns for the two portfolio strategies, "MVP-C" and "GMV-C." Which strategy has the higher Sharpe ratio? Why?*

In [ ]:
table2 = summarize(oos2, {"MVP-C": W_mvpc, "GMV-C": W_gmvc})
display(table2.round(3))
best2 = table2["Sharpe (ann.)"].idxmax()
z, p = jkm_test(oos2["GMV-C"], oos2["MVP-C"])
print(f"Higher out-of-sample Sharpe ratio: {best2}")
print(f"JKM test GMV-C vs MVP-C: z = {z:.2f}, p-value = {p:.3f}")

**Answer.** The table reports the Sharpe ratios of the two constrained strategies and which one is higher. Interpretation:

* **GMV-C** does not use expected returns at all and the short-sale constraint acts as a **shrinkage of the covariance matrix**: the constrained GMV equals the unconstrained GMV computed with V̂_JM = V̂ − λ1' − 1λ', where λₙ > 0 only for the stocks whose constraint binds. Their covariances, which are often over-estimated, are reduced (Jagannathan and Ma 2003). The result is a low-volatility, stable portfolio with low turnover.
* **MVP-C** still relies on the noisy sample means. The constraint limits the damage (it is equivalent to raising the means of the stocks the optimizer wanted to short, μ + λ, i.e., shrinking the means towards the average), but the portfolio remains concentrated in the stocks with the highest past average returns, which are largely noise.

So, in general, we expect GMV-C to have the higher Sharpe ratio, and in any case lower volatility and turnover. If in our sample MVP-C has the higher Sharpe ratio, it means that the stocks with high past average returns kept doing well over 2005–2022 (momentum in a few names); the JKM p-value shows whether the difference is statistically significant, and with only 10 stocks and one sample period such a result should not be generalized.

## Q2.5
*Finally, out of the four strategies you have considered in the two questions, "MVP," "GMV," "MVP-C," and "GMV-C," which strategy has the highest Sharpe ratio. Why do you think this strategy performs the best?*

In [ ]:
oos_all = pd.concat([oos1, oos2], axis=1)
W_all = {"MVP": W_mvp, "GMV": W_gmv, "MVP-C": W_mvpc, "GMV-C": W_gmvc}
table_all = summarize(oos_all, W_all)

# Benchmark (not required): the 1/N portfolio, rebalanced monthly
ew = R_oos.mean(axis=1)
table_all.loc["1/N (benchmark)"] = [12 * ew.mean(), np.sqrt(12) * ew.std(), sharpe(ew), 1 / N, np.nan]
display(table_all.round(3))

four = table_all.loc[["MVP", "GMV", "MVP-C", "GMV-C"], "Sharpe (ann.)"]
best = four.idxmax()
print(f"Highest out-of-sample Sharpe ratio among the four strategies: {best} ({four.max():.3f})")
print("Ranking:", " > ".join(four.sort_values(ascending=False).index))
for other in four.index.drop(best):
    z, p = jkm_test(oos_all[best], oos_all[other])
    print(f"  {best} vs {other}: z = {z:.2f}, p = {p:.3f}")

ax = table_all["Sharpe (ann.)"].plot.bar(title="Out-of-sample Sharpe ratios (annualized)")
ax.axhline(0, color="k", lw=0.8); plt.xticks(rotation=0); plt.show()

**Answer.** The output above gives the strategy with the highest out-of-sample Sharpe ratio and the full ranking (with the 1/N portfolio as an additional benchmark). The general pattern, which matches the evidence in Lecture 4 (DeMiguel, Garlappi and Uppal 2009; Jagannathan and Ma 2003), is:

1. **Ignoring the means pays.** Strategies that do not use μ̂ (GMV, GMV-C) perform better than those that do (MVP, MVP-C), because expected returns cannot be estimated precisely (standard error σ/√T, independent of the sampling frequency), while variances and covariances can be estimated much more precisely.
2. **Constraints act as shrinkage.** The non-negativity constraint limits extreme positions. For MVP it shrinks the means, for GMV it shrinks the covariance matrix. It removes the long–short bets that exploit estimation error, and it reduces volatility and turnover dramatically (compare "avg max |w|" and "avg turnover").
3. **The unconstrained MVP is the worst**: it combines noisy means, an inverse covariance matrix that amplifies errors, and the instability of the rescaling by 1'V̂⁻¹μ̂.

Hence the best strategy is typically **GMV-C** (or GMV): it avoids the main source of estimation error and regularizes the covariance matrix. If our ranking differs, the reason is sample-specific: only 10 stocks, one historical period (2005–2022, including the 2008 crisis and Covid), and Sharpe ratio differences that are often not statistically significant (see the JKM p-values).

**Caveats.** (i) The 10 stocks are today's S&P 500 constituents, so the sample has survivorship bias. (ii) No transaction costs are included; with costs, the ranking would favour the low-turnover strategies (GMV-C, 1/N) even more. (iii) The JKM test assumes IID Normal returns; a block bootstrap (Ledoit and Wolf 2008) would be more robust.